In [6]:
import os
import cv2
import pickle
import mediapipe as mp

# إعداد MediaPipe
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5)

# إعداد مجلد البيانات
DATA_DIR = './new_sign_language_data'  # تغيير اسم المجلد
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

# تحميل القاموس القديم إن وجد
labels_dict = {}
if os.path.exists('labels_dict.pickle'):
    with open('labels_dict.pickle', 'rb') as f:
        labels_dict = pickle.load(f)

# إعداد الكاميرا
cap = cv2.VideoCapture(0)

# إعداد معلمات التسجيل
NUM_VIDEOS = 5  # عدد الفيديوهات لكل فئة
VIDEO_DURATION = 5  # مدة كل فيديو بالثواني
FPS = 20  # عدد الإطارات في الثانية

def record_video_for_class(class_name, arabic_label):
    """تسجيل فيديوهات للفئة المحددة مع حفظ الميزات."""
    # إنشاء المجلد الخاص بالفئة
    class_dir = os.path.join(DATA_DIR, class_name)
    if not os.path.exists(class_dir):
        os.makedirs(class_dir)

    # إضافة التسمية العربية إلى القاموس
    labels_dict[class_name] = arabic_label

    print(f"Recording {NUM_VIDEOS} videos for class '{class_name}'...")

    for video_idx in range(NUM_VIDEOS):
        video_features = []
        print(f"Recording video {video_idx + 1} of {NUM_VIDEOS}...")

        # إنشاء مسار الفيديو
        video_path = os.path.join(class_dir, f'video_{video_idx + 1}.avi')
        video_writer = cv2.VideoWriter(video_path, cv2.VideoWriter_fourcc(*'XVID'), FPS, (640, 480))

        frame_count = 0
        while frame_count < VIDEO_DURATION * FPS:
            ret, frame = cap.read()
            if not ret:
                print("Failed to capture frame.")
                break

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(frame_rgb)

            # استخراج الميزات
            frame_features = extract_features(results)
            video_features.append(frame_features)

            # كتابة الإطار في الفيديو
            video_writer.write(frame)

            # عرض الإطار مع المعالم
            annotated_frame = annotate_frame(frame, results)
            cv2.imshow('frame', annotated_frame)

            frame_count += 1

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        video_writer.release()

        # حفظ ميزات الفيديو
        feature_path = os.path.join(class_dir, f'features_{video_idx + 1}.pickle')
        with open(feature_path, 'wb') as f:
            pickle.dump(video_features, f)

        print(f"Video {video_idx + 1} and features saved for class '{class_name}'.")

def extract_features(results):
    """استخراج ميزات الإطار بناءً على نتائج MediaPipe."""
    frame_features = []

    # معالم اليد اليمنى
    if results.right_hand_landmarks:
        for landmark in results.right_hand_landmarks.landmark:
            frame_features.extend([landmark.x, landmark.y, landmark.z])
    else:
        frame_features.extend([0] * 63)  # 21 نقطة * 3 أبعاد

    # معالم اليد اليسرى
    if results.left_hand_landmarks:
        for landmark in results.left_hand_landmarks.landmark:
            frame_features.extend([landmark.x, landmark.y, landmark.z])
    else:
        frame_features.extend([0] * 63)

    # معالم الجسم
    if results.pose_landmarks:
        for landmark in results.pose_landmarks.landmark:
            frame_features.extend([landmark.x, landmark.y, landmark.z])
    else:
        frame_features.extend([0] * 99)  # 33 نقطة * 3 أبعاد

    return frame_features

def annotate_frame(frame, results):
    """رسم المعالم على الإطار لعرضها أثناء التسجيل."""
    annotated_frame = frame.copy()
    mp.solutions.drawing_utils.draw_landmarks(annotated_frame, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    mp.solutions.drawing_utils.draw_landmarks(annotated_frame, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    mp.solutions.drawing_utils.draw_landmarks(annotated_frame, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
    return annotated_frame

while True:
    class_name = input("Enter the class name (in English, or type 'done' to finish): ")
    if class_name.lower() == 'done':
        break

    arabic_label = input(f"Enter the Arabic word for the class '{class_name}': ")
    record_video_for_class(class_name, arabic_label)

cap.release()
cv2.destroyAllWindows()

# حفظ القاموس المحدث
with open('labels_dict.pickle', 'wb') as f:
    pickle.dump(labels_dict, f)

print("Data collection completed and labels saved successfully.")

Enter the class name (in English, or type 'done' to finish):  شكرا
Enter the Arabic word for the class 'شكرا':  شكرا


Recording 5 videos for class 'شكرا'...
Recording video 1 of 5...
Video 1 and features saved for class 'شكرا'.
Recording video 2 of 5...
Video 2 and features saved for class 'شكرا'.
Recording video 3 of 5...
Video 3 and features saved for class 'شكرا'.
Recording video 4 of 5...
Video 4 and features saved for class 'شكرا'.
Recording video 5 of 5...
Video 5 and features saved for class 'شكرا'.


Enter the class name (in English, or type 'done' to finish):  الحمد لله
Enter the Arabic word for the class 'الحمد لله':  الحم


Recording 5 videos for class 'الحمد لله'...
Recording video 1 of 5...
Video 1 and features saved for class 'الحمد لله'.
Recording video 2 of 5...
Video 2 and features saved for class 'الحمد لله'.
Recording video 3 of 5...
Video 3 and features saved for class 'الحمد لله'.
Recording video 4 of 5...
Video 4 and features saved for class 'الحمد لله'.
Recording video 5 of 5...
Video 5 and features saved for class 'الحمد لله'.


Enter the class name (in English, or type 'done' to finish):  كيف الحال
Enter the Arabic word for the class 'كيف الحال':  ك


...

Recording 5 videos for class 'نحن'...
Recording video 1 of 5...
Video 1 and features saved for class 'نحن'.
Recording video 2 of 5...
Video 2 and features saved for class 'نحن'.
Recording video 3 of 5...
Video 3 and features saved for class 'نحن'.
Recording video 4 of 5...
Video 4 and features saved for class 'نحن'.
Recording video 5 of 5...
Video 5 and features saved for class 'نحن'.


Enter the class name (in English, or type 'done' to finish):  مع السلامة
Enter the Arabic word for the class 'مع السلامة':  مع السلامة


Recording 5 videos for class 'مع السلامة'...
Recording video 1 of 5...
Video 1 and features saved for class 'مع السلامة'.
Recording video 2 of 5...
Video 2 and features saved for class 'مع السلامة'.
Recording video 3 of 5...
Video 3 and features saved for class 'مع السلامة'.
Recording video 4 of 5...
Video 4 and features saved for class 'مع السلامة'.
Recording video 5 of 5...
Video 5 and features saved for class 'مع السلامة'.


Enter the class name (in English, or type 'done' to finish):  done


Data collection completed and labels saved successfully.


In [1]:
import os
import pickle
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# إعداد المجلد الذي يحتوي على البيانات
DATA_DIR = './new_sign_language_data'  # اسم المجلد الذي يحتوي على البيانات
X, y = [], []

# تحميل الميزات من الفيديوهات المحفوظة
for class_name in os.listdir(DATA_DIR):
    class_dir = os.path.join(DATA_DIR, class_name)
    if not os.path.isdir(class_dir):
        continue

    for feature_file in os.listdir(class_dir):
        if feature_file.endswith('.pickle'):
            feature_path = os.path.join(class_dir, feature_file)
            with open(feature_path, 'rb') as f:
                video_features = pickle.load(f)
                X.extend(video_features)
                y.extend([class_name] * len(video_features))

# تحويل القوائم إلى numpy arrays
X = np.array(X)
y = np.array(y)

# تحويل التسميات إلى ترميز رقمي
class_names = sorted(list(set(y)))  # قائمة من جميع الفئات الفريدة
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
y = np.array([class_to_idx[label] for label in y])

# تحويل y إلى ترميز one-hot
y = to_categorical(y, num_classes=len(class_names))

# تقسيم البيانات إلى تدريب واختبار
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# تحويل البيانات لتناسب RNN (الشكل يجب أن يكون [عدد الأمثلة, عدد الإطارات, عدد الميزات لكل إطار])
X_train = X_train.reshape((X_train.shape[0], -1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], -1, X_test.shape[1]))

print(f"Training data shape: {X_train.shape}, Test data shape: {X_test.shape}")
# بناء نموذج مدمج باستخدام LSTM و MLP
model = Sequential()

# إضافة طبقة LSTM لتحليل التسلسل الزمني
model.add(LSTM(128, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False))  # 128 هو عدد الخلايا في LSTM

# إضافة Dropout لتقليل الإفراط في التعميم
model.add(Dropout(0.2))

# إضافة طبقة MLP (طبقة مخفية)
model.add(Dense(64, activation='relu'))

# إضافة طبقة مخفية أخرى في MLP
model.add(Dense(32, activation='relu'))

# إضافة طبقة الإخراج (تصنيف متعدد الفئات)
model.add(Dense(len(class_names), activation='softmax'))

# تجميع النموذج
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# عرض ملخص النموذج
model.summary()
# تدريب النموذج
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))

# حفظ النموذج المدرب
model.save('sign_language_mlp_rnn_model.h5')
# تقييم النموذج
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")

Training data shape: (7200, 1, 225), Test data shape: (1800, 1, 225)
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 128)               181248    
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense (Dense)               (None, 64)                8256      
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 18)                594       
                                                                 
Total params: 192178 (750.70 KB)
Trainable params: 192178 (750.70 KB)
Non-trainable params: 0 (0.00 Byte)
_____________

C:\Users\USER\AppData\Local\anaconda3\envs\myenv\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


57/57 [==============================] - 0s 2ms/step - loss: 0.2662 - accuracy: 0.9244
Test accuracy: 0.9244


In [2]:
import cv2
import numpy as np
import pickle
import mediapipe as mp
from tensorflow.keras.models import load_model
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display
from gtts import gTTS
import pygame
import time
import os
from collections import Counter
import tkinter as tk
from tkinter import messagebox

import pygame.mixer


import tempfile
import shutil

# إعداد MediaPipe
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5)

# تحميل النموذج المدرب
model = load_model('sign_language_mlp_rnn_model.h5')

# تحميل القاموس الذي يحتوي على المترجمات بين الفئات
with open('labels_dict.pickle', 'rb') as f:
    labels_dict = pickle.load(f)

# إعداد محرك تحويل النص إلى صوت باستخدام gTTS
pygame.mixer.init()

# متغيرات لحفظ الكلمات
current_word = ""
current_words = []

def extract_features(results):
    frame_features = []
    if results.right_hand_landmarks:
        for landmark in results.right_hand_landmarks.landmark:
            frame_features.extend([landmark.x, landmark.y, landmark.z])
    else:
        frame_features.extend([0] * 63)

    if results.left_hand_landmarks:
        for landmark in results.left_hand_landmarks.landmark:
            frame_features.extend([landmark.x, landmark.y, landmark.z])
    else:
        frame_features.extend([0] * 63)

    if results.pose_landmarks:
        for landmark in results.pose_landmarks.landmark:
            frame_features.extend([landmark.x, landmark.y, landmark.z])
    else:
        frame_features.extend([0] * 99)

    return frame_features

def put_arabic_text(image, text, position, font_path="arial.ttf", font_size=32, color=(0, 255, 0)):
    reshaped_text = arabic_reshaper.reshape(text)
    bidi_text = get_display(reshaped_text)
    img_pil = Image.fromarray(image)
    draw = ImageDraw.Draw(img_pil)
    font = ImageFont.truetype(font_path, font_size)
    draw.text(position, bidi_text, font=font, fill=color)
    return np.array(img_pil)

# بدء التسجيل لمدة معينة (ثواني)
record_duration = 10  # مدة التسجيل بالثواني
frame_rate = 30  # عدد الإطارات في الثانية
total_frames = record_duration * frame_rate

start_time = time.time()
is_recording = False  # لا نبدأ التسجيل إلا بعد ظهور اليد
predictions = []  # لتخزين التنبؤات أثناء التسجيل
no_sign_detected = True  # لتتبع ما إذا كان تم اكتشاف أي إشارة أم لا

# لتتبع حالة اليد
hand_detected = False

def start_recording():
    """بدء التسجيل وتشغيل الكاميرا"""
    global is_recording, predictions, no_sign_detected, hand_detected, start_time
    cap = cv2.VideoCapture(0)

    predictions = []  # إعادة تعيين التنبؤات
    no_sign_detected = True
    hand_detected = False
    start_time = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(frame_rgb)

        # التحقق من وجود اليد
        if results.right_hand_landmarks or results.left_hand_landmarks:
            hand_detected = True  # عندما تظهر اليد

        if hand_detected:
            # إذا كانت اليد موجودة، نبدأ في التسجيل والتنبؤ
            features = extract_features(results)

            if not results.right_hand_landmarks and not results.left_hand_landmarks:
                arabic_label = "لم يتم التعرف"
                no_sign_detected = True
            else:
                # إعادة تشكيل الميزات لتناسب شكل الإدخال المتوقع من النموذج
                features = np.array([features])
                features = features.reshape((features.shape[0], 1, -1))  # إضافة بعد جديد لتطابق الشكل (1, 1, 225)

                prediction = model.predict(features)
                predicted_class_idx = np.argmax(prediction)
                predicted_class_name = class_names[predicted_class_idx]
                arabic_label = labels_dict.get(predicted_class_name, "غير متعرف عليها")
                no_sign_detected = False

            predictions.append(arabic_label)

            annotated_frame = frame.copy()
            mp.solutions.drawing_utils.draw_landmarks(annotated_frame, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
            mp.solutions.drawing_utils.draw_landmarks(annotated_frame, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
            mp.solutions.drawing_utils.draw_landmarks(annotated_frame, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)

            annotated_frame = put_arabic_text(annotated_frame, arabic_label, (50, 50), font_path="arial.ttf", font_size=32)
            cv2.imshow('frame', annotated_frame)

            # التحقق من الوقت لوقف التسجيل
            elapsed_time = time.time() - start_time
            if elapsed_time >= record_duration:
                is_recording = False
                break

        else:
            # إذا لم تكن اليد موجودة، عرض الكاميرا بدون تسجيل أو تنبؤ
            cv2.imshow('frame', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    # عند انتهاء التسجيل، نعرض التنبؤ النهائي
    if predictions:
        most_common_prediction, _ = Counter(predictions).most_common(1)[0]

        if no_sign_detected:
            most_common_prediction = "لم يتم التعرف"

        global current_word
        current_word = most_common_prediction
        current_words.append(current_word)

        sound_file = f"output_{most_common_prediction}.mp3"
        if not os.path.exists(sound_file):
            tts = gTTS(text=most_common_prediction, lang='ar')
            tts.save(sound_file)

        pygame.mixer.music.load(sound_file)
        pygame.mixer.music.play()

        print(f"التنبؤ النهائي: {most_common_prediction}")

    # تحديث النصوص في واجهة Tkinter
    update_display()

def update_display():
    """تحديث الخانات لعرض الكلمة الحالية والجملة الحالية"""
    word_label.config(text=f"الكلمة الحالية: {current_word}")
    sentence_label.config(text=f"الجملة الحالية: {' '.join(current_words)}")

# الآن ننشئ واجهة المستخدم الرسومية باستخدام Tkinter

def speak_current_word():
    """نطق الكلمة المخزنة في current_word"""
    if current_word:
        # دمج الكلمات معًا
        word_text = current_word
        
        # تحديد مسار المجلد الذي سيتم حفظ الملف فيه
        file_path = os.path.join(os.getcwd(), "current_word.mp3")
        
        # التحقق من وجود الملف وحذفه إذا كان موجودًا
        if os.path.exists(file_path):
            os.remove(file_path)
        
        # تحويل النص إلى صوت وحفظه
        tts = gTTS(text=word_text, lang='ar')
        tts.save(file_path)
        
        # تحميل وتشغيل الصوت
        pygame.mixer.music.load(file_path)
        pygame.mixer.music.play()

        # انتظر حتى ينتهي الصوت قبل السماح بتشغيله مرة أخرى
        while pygame.mixer.music.get_busy():
            time.sleep(0.1)  # انتظر بشكل دوري

        print(f"تم نطق الكلمة: {word_text}")




def speak_current_words():
    """نطق الكلمات المخزنة في current_words"""
    if current_words:
        # دمج الكلمات معًا
        words_text = " ".join(current_words)
        
        # إنشاء ملف صوت مؤقت
        temp_dir = tempfile.mkdtemp()  # إنشاء مجلد مؤقت
        file_path = os.path.join(temp_dir, "current_words.mp3")

        # تحويل النص إلى صوت وحفظه في المجلد المؤقت
        tts = gTTS(text=words_text, lang='ar')
        tts.save(file_path)
        
        # تهيئة pygame والموسيقى
        pygame.mixer.init()
        pygame.mixer.music.load(file_path)
        pygame.mixer.music.play()

        # انتظر حتى ينتهي الصوت من التشغيل
        while pygame.mixer.music.get_busy():
            time.sleep(0.1)  # انتظر حتى تنتهي الموسيقى

        # نقل الملف إلى المجلد النهائي بعد الانتهاء من تشغيله
        final_path = os.path.join(os.getcwd(), "current_words.mp3")
        
        # محاولة نقل الملف بعد التأكد من أن الصوت قد انتهى
        try:
            shutil.move(file_path, final_path)  # نقل الملف إلى المجلد النهائي
            print(f"تم نقل الملف إلى {final_path}")
        except PermissionError:
            print("فشل في نقل الملف: الملف لا يزال قيد الاستخدام.")
        
        # حذف المجلد المؤقت مع محتوياته
        try:
            shutil.rmtree(temp_dir)  # حذف المجلد المؤقت وكل محتوياته
            print("تم حذف المجلد المؤقت بنجاح.")
        except OSError as e:
            print(f"فشل في حذف المجلد المؤقت: {e}")

        print(f"تم نطق الجملة: {words_text}")
    else:
        print("لا توجد كلمات للنطق.")




def reset_words():
    """إعادة تعيين الكلمات"""
    global current_word, current_words
    current_word = ""
    current_words = []
    update_display()
    messagebox.showinfo("إعادة تعيين", "تم إعادة تعيين الكلمات.")

# إعداد واجهة المستخدم باستخدام Tkinter
root = tk.Tk()
root.title("تطبيق التعرف على لغة الإشارة")

# عرض الكلمة الحالية
word_label = tk.Label(root, text="الكلمة الحالية: " + current_word, font=("Arial", 16))
word_label.pack(pady=10)

# عرض الجملة الحالية
sentence_label = tk.Label(root, text="الجملة الحالية: " + " ".join(current_words), font=("Arial", 16))
sentence_label.pack(pady=10)

# زر نطق الكلمة المخزنة في current_word
speak_button = tk.Button(root, text="نطق الكلمة المخزنة", command=speak_current_word, font=("Arial", 14))
speak_button.pack(pady=10)

# زر نطق جميع الكلمات المخزنة في current_words
speak_words_button = tk.Button(root, text="نطق جميع الكلمات", command=speak_current_words, font=("Arial", 14))
speak_words_button.pack(pady=10)

# زر إعادة تعيين الكلمات
reset_button = tk.Button(root, text="إعادة تعيين", command=reset_words, font=("Arial", 14))
reset_button.pack(pady=10)

# زر لتشغيل الكاميرا وبدء التسجيل
record_button = tk.Button(root, text="تسجيل", command=start_recording, font=("Arial", 14))
record_button.pack(pady=10)

# بدء واجهة المستخدم
root.mainloop()

C:\Users\USER\AppData\Local\anaconda3\envs\myenv\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.10.19)
Hello from the pygame community. https://www.pygame.org/contribute.html
1/1 [==============================] - 0s 23ms/step
التنبؤ النهائي: نحن


In [3]:
from fastapi import FastAPI
import numpy as np
import pickle
from tensorflow.keras.models import load_model

app = FastAPI()

# تحميل الموديل
model = load_model('sign_language_mlp_rnn_model.h5')

# تحميل الليبلز
with open('labels_dict.pickle', 'rb') as f:
    labels_dict = pickle.load(f)

class_names = list(labels_dict.keys())

@app.get("/")
def home():
    return {"message": "API شغال 🚀"}

@app.post("/predict")
def predict(data: dict):
    try:
        features = np.array(data["features"])
        features = features.reshape((1, 1, -1))

        prediction = model.predict(features)
        idx = np.argmax(prediction)

        class_name = class_names[idx]
        arabic_label = labels_dict.get(class_name, "غير معروف")

        return {
            "prediction": class_name,
            "arabic": arabic_label
        }

    except Exception as e:
        return {"error": str(e)}

In [4]:
!pip install nest-asyncio

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()

uvicorn.run("main:app", host="127.0.0.1", port=8001, reload=True)

INFO:     Will watch for changes in these directories: ['C:\\Users\\USER\\Desktop\\Project Sign']
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)
INFO:     Started reloader process [1312] using StatReload
